# Vũ Văn Học 2A202600653
# Lab Day 19: Xây dựng hệ thống GraphRAG với Tech Company Corpus

## Phần 1: Nghiên cứu và Chuẩn bị (Theory)

**1. Entity Extraction: Làm sao để LLM phân biệt được đâu là thực thể (Node) và đâu là thuộc tính?**
LLM phân biệt được thực thể và thuộc tính chủ yếu dựa vào **Prompt (chỉ thị)** mà chúng ta cung cấp: Thực thể (Node) là các danh từ chỉ người, tổ chức, sự kiện độc lập. Thuộc tính (Property) là những thông tin mô tả chi tiết, giá trị gắn liền với thực thể (như tuổi, năm thành lập). Kỹ sư thường dùng Few-shot Prompting hoặc định nghĩa JSON Schema cụ thể để điều hướng LLM.

**2. Graph Construction: Tại sao việc khử trùng lặp (Deduplication) lại quan trọng trong đồ thị?**
Nếu không khử trùng lặp, đồ thị sẽ có nhiều node rời rạc biểu diễn cùng một thực thể (VD: "OpenAI", "công ty OpenAI"), làm đứt gãy kết nối thông tin. Hợp nhất thực thể giúp tạo ra một đồ thị liên kết chặt chẽ và nhất quán, hỗ trợ suy luận đa bước.

**3. Query Answering: Sự khác biệt giữa duyệt đồ thị theo chiều rộng (BFS) và tìm kiếm vector thông thường là gì?**
- **Tìm kiếm vector (Flat RAG):** Tìm kiếm các đoạn văn bản (chunks) có ngữ nghĩa gần nhất thông qua khoảng cách vector. Dễ bỏ sót thông tin liên kết ẩn (multi-hop).
- **Duyệt đồ thị BFS (GraphRAG):** Bắt đầu từ node thực thể chính được trích xuất từ câu hỏi, BFS duyệt qua các node lân cận (cách 1 hop, 2 hops) để thu thập ngữ cảnh, rất hiệu quả để tổng hợp thông tin phân tán.


## Phần 2 & 3: Thực hành Xây dựng Hệ thống

In [ ]:
# !pip install networkx matplotlib neo4j openai pandas
# !pip install langchain langchain-openai langchain-community faiss-cpu sentence-transformers

### Bước 1: Trích xuất thực thể và quan hệ (Indexing)
Sử dụng LLM để đọc bộ dữ liệu text từ thư mục `dataset` và trích xuất thành các bộ ba (triples).

In [ ]:
import dotenv
import os
import glob
import json
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

# Khởi tạo cấu hình API
dotenv.load_dotenv()
API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL")
MODEL_NAME = os.getenv("MODEL_NAME")

llm = ChatOpenAI(temperature=0, model=MODEL_NAME, api_key=API_KEY, base_url=BASE_URL)

prompt_template = """
Trích xuất thông tin từ đoạn văn bản sau thành các bộ ba (triples) để xây dựng Knowledge Graph.
Lưu ý quan trọng: Các mốc thời gian, số liệu (năm, doanh thu, tỷ lệ...) KHÔNG được làm Object độc lập mà phải được lưu dưới dạng metadata (thuộc tính) của quan hệ.
Trả về kết quả dưới dạng danh sách các JSON object. Mỗi object có cấu trúc chính xác như sau: {{"subject": "...", "relation": "...", "object": "...", "metadata": {{"key": "value"}}}}.

Chỉ trả về chuỗi JSON hợp lệ, không giải thích gì thêm.

Văn bản: {text}
"""
prompt = PromptTemplate(template=prompt_template, input_variables=["text"])
chain = prompt | llm

dataset_dir = "./dataset"
all_triples = []

# Load 3 file đại diện để test (ví dụ)
file_paths = ["dataset/doc_1.txt", "dataset/doc_13.txt", "dataset/doc_46.txt"]
texts = []

for file_path in file_paths:
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()[:1500] # Giới hạn token
            texts.append(text)
            
            response = chain.invoke({"text": text})
            print(f"--- Triples từ {os.path.basename(file_path)} ---")
            
            # Parse JSON từ LLM
            try:
                content = response.content.strip()
                if content.startswith("```json"):
                    content = content[7:-3]
                elif content.startswith("```"):
                    content = content[3:-3]
                
                triples_data = json.loads(content)
                for item in triples_data:
                    if "subject" in item and "relation" in item and "object" in item:
                        all_triples.append((item["subject"], item["relation"], item["object"], item.get("metadata", {})))
            except Exception as e:
                pass

print(f"\nTổng số triples trích xuất được: {len(all_triples)}")

### Bước 2: Xây dựng đồ thị (Construction)
Sử dụng NetworkX để dựng đồ thị từ danh sách `all_triples`.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()

# Thêm các edges và metadata
for subject, relation, obj, metadata in all_triples:
    G.add_edge(subject, obj, label=relation, **metadata)

plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, k=0.5)
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=1500, font_size=8, font_weight='bold', arrows=True)

edge_labels = nx.get_edge_attributes(G, 'label')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red', font_size=7)

plt.title("Tech Company Graph from Real Dataset")
plt.show()

### Bước 3 & 4: Thực thi truy vấn (Querying) & So sánh (Evaluation)
So sánh kết quả truy vấn giữa **GraphRAG** và **Flat RAG**.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# --- 1. SETUP FLAT RAG ---
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_texts(texts, embeddings)

# --- 2. SETUP QUERIES ---
queries = [
    ("VinFast", "What is VinFast's ticker symbol and its Q3 2024 revenue increase?"),
    ("electric vehicle", "How much has the US electric vehicle market grown from 2018 to 2020?"),
    ("VinFast", "Who is the Chairwoman of VinFast?"),
    ("OpenAI", "Who founded OpenAI?"),
    ("Energy Information Administration", "What sectors does the Total Energy Monthly Data cover?")
]

# --- 3. EXECUTE & COMPARE ---
for query_node, query_question in queries:
    print(f"\n{'='*50}")
    print(f"Q: {query_question}")
    
    # --- FLAT RAG ---
    docs = vectorstore.similarity_search(query_question, k=2)
    flat_context = "\n".join([d.page_content for d in docs])
    flat_prompt = f"Dựa vào thông tin sau:\n{flat_context}\nTrả lời câu hỏi: {query_question}"
    flat_ans = llm.invoke(flat_prompt).content.strip()
    print(f"\n[Flat RAG Answer]:\n{flat_ans}")
    
    # --- GRAPH RAG ---
    # Tìm node gốc
    nodes_to_check = [n for n in G.nodes if n.lower() in query_node.lower() or query_node.lower() in n.lower()]
    if not nodes_to_check:
        print(f"\n[GraphRAG Answer]:\nKhông tìm thấy thực thể {query_node} trong đồ thị.")
        continue
        
    context_triples = []
    for node in nodes_to_check:
        for neighbor in G.successors(node):
            edge_data = G.get_edge_data(node, neighbor)
            relation = edge_data.get('label', '')
            meta_strs = [f"{k}: {v}" for k, v in edge_data.items() if k != 'label']
            meta_context = f" ({', '.join(meta_strs)})" if meta_strs else ""
            context_triples.append(f"{node} {relation} {neighbor}{meta_context}")
    
    graph_context = ". ".join(context_triples)
    graph_prompt = f"Dựa vào thông tin sau từ Knowledge Graph: '{graph_context}'. Hãy trả lời câu hỏi: {query_question}"
    graph_ans = llm.invoke(graph_prompt).content.strip()
    print(f"\n[GraphRAG Answer]:\n{graph_ans}")
    print("="*50)